## 10 - Teste sintético YOLO protegido e comparação com Hough (v3)
Este notebook abre o teste sintético uma única vez, depois de família, pesos,
threshold, matching e regras estatísticas terem sido congelados no Notebook 09.
Os três checkpoints YOLO11n-OBB são avaliados; o resultado primário é a média entre
seeds. Nenhuma seed, threshold ou regra poderá ser alterada após observar o teste.
O evento é retomável: uma interrupção pode continuar o mesmo event_id, reutilizando
somente seeds já concluídas e verificadas por hash. Um evento finalizado jamais executa
inferência novamente. ASTA permanece externo e intocado neste notebook.

In [1]:
import json, platform, sys, torch

if not torch.cuda.is_available():
    raise RuntimeError(
        'GPU obrigatória. No Colab, selecione Ambiente de execução > Alterar tipo de '
        'ambiente > GPU e execute novamente a partir desta célula.'
    )
props=torch.cuda.get_device_properties(0)
livre,total=torch.cuda.mem_get_info(0)
relatorio_gpu={
    'status':'GPU_APROVADA','gpu':torch.cuda.get_device_name(0),
    'cuda':torch.version.cuda,'torch':torch.__version__,
    'vram_total_gb':round(total/1024**3,2),'vram_livre_gb':round(livre/1024**3,2),
    'python':sys.version.split()[0],'plataforma':platform.platform()
}
assert total>=8*1024**3,'GPU com menos de 8 GiB não foi aprovada para este protocolo.'
print(json.dumps(relatorio_gpu,indent=2,ensure_ascii=False))

{
  "status": "GPU_APROVADA",
  "gpu": "Tesla T4",
  "cuda": "12.8",
  "torch": "2.11.0+cu128",
  "vram_total_gb": 14.56,
  "vram_livre_gb": 14.46,
  "python": "3.13.15",
  "plataforma": "Linux-6.6.122+-x86_64-with-glibc2.35"
}


## 1. Ambiente fixado e restauração íntegra pelo TAR
Execute em GPU. Esta restauração é idêntica à dos Notebooks 08–09 e verifica o pacote,
o manifest e os conjuntos exatos de imagens/labels antes de qualquer inferência.

In [2]:
%pip install -q ultralytics==8.4.127 opencv-python-headless pandas pyyaml tqdm

from google.colab import drive
from pathlib import Path
import hashlib, json, math, os, shutil, tarfile, tempfile, time
import cv2
import numpy as np
import pandas as pd
import yaml
from tqdm.auto import tqdm

def log_etapa(mensagem):
    print(f"[{time.strftime('%H:%M:%S')}] {mensagem}",flush=True)

def sha256_arquivo(caminho,bloco=8*1024*1024):
    h=hashlib.sha256()
    with open(caminho,'rb') as arq:
        for parte in iter(lambda:arq.read(bloco),b''):h.update(parte)
    return h.hexdigest()

def sha256_com_progresso(caminho,descricao,bloco=8*1024*1024):
    caminho=Path(caminho);total=caminho.stat().st_size;h=hashlib.sha256()
    with caminho.open('rb') as arq,tqdm(total=total,desc=descricao,unit='B',unit_scale=True) as barra:
        for parte in iter(lambda:arq.read(bloco),b''):
            h.update(parte);barra.update(len(parte))
    return h.hexdigest()

def copiar_arquivo_com_progresso(origem,destino,descricao,bloco=8*1024*1024):
    origem,destino=Path(origem),Path(destino);destino.parent.mkdir(parents=True,exist_ok=True)
    total=origem.stat().st_size
    with origem.open('rb') as entrada,destino.open('wb') as saida, \
         tqdm(total=total,desc=descricao,unit='B',unit_scale=True) as barra:
        for parte in iter(lambda:entrada.read(bloco),b''):
            saida.write(parte);barra.update(len(parte))

def copiar_arvore_com_progresso(origem,destino,descricao):
    origem,destino=Path(origem),Path(destino)
    arquivos=sorted(p for p in origem.rglob('*') if p.is_file())
    for arquivo in tqdm(arquivos,desc=descricao,unit='arq'):
        alvo=destino/arquivo.relative_to(origem);alvo.parent.mkdir(parents=True,exist_ok=True)
        shutil.copy2(arquivo,alvo)

drive.mount('/content/drive',force_remount=True)
BACKUP_DIR=Path('/content/drive/MyDrive/tcc-satellite-streaks')
RUN_ID=(BACKUP_DIR/'data/synthetic_runs/LATEST_RUN_V3.txt').read_text(encoding='utf-8').strip()
PACOTES_DRIVE=BACKUP_DIR/'data/yolo_packages'/RUN_ID
contrato_pacote=json.loads((PACOTES_DRIVE/'contrato_pacote_yolo_v3.json').read_text(encoding='utf-8'))
assert contrato_pacote['run_id']==RUN_ID
pacote_remoto=PACOTES_DRIVE/contrato_pacote['arquivo']
pacote_local=Path('/content')/contrato_pacote['arquivo']

if not pacote_local.exists():
    copiar_arquivo_com_progresso(pacote_remoto,pacote_local,'Restaurando TAR do dataset')
hash_pacote=sha256_com_progresso(pacote_local,'Verificando TAR local')
assert hash_pacote==contrato_pacote['sha256_pacote']

raiz_runs=Path('/content/data/synthetic_runs');raiz_runs.mkdir(parents=True,exist_ok=True)
RUN=raiz_runs/RUN_ID
if RUN.exists():
    assert (RUN/'manifest_geracao_v3.csv').is_file()
    assert sha256_arquivo(RUN/'manifest_geracao_v3.csv')==contrato_pacote['sha256_manifest']
    print('RUN local íntegro reutilizado.')
else:
    temp=Path(tempfile.mkdtemp(prefix='extracao_yolo_',dir='/content'))
    try:
        with tarfile.open(pacote_local,'r') as tar:
            membros=tar.getmembers()
            assert len(membros)==contrato_pacote['membros']
            for membro in membros:
                partes=Path(membro.name).parts
                assert partes and partes[0]==RUN_ID and '..' not in partes
                assert not membro.issym() and not membro.islnk()
            for membro in tqdm(membros,desc='Extraindo dataset local',unit='arq'):
                tar.extract(membro,path=temp,filter='data')
        shutil.move(str(temp/RUN_ID),str(RUN))
    finally:
        if temp.exists():shutil.rmtree(temp)

df=pd.read_csv(RUN/'manifest_geracao_v3.csv')
HASH_MANIFEST=sha256_arquivo(RUN/'manifest_geracao_v3.csv')
validacao=json.loads((RUN/'resumo_validacao_v3.json').read_text(encoding='utf-8'))
assert len(df)==6000 and HASH_MANIFEST==contrato_pacote['sha256_manifest']
assert validacao.get('aprovado_para_treino') is True
for split in ['train','val','test']:
    esperado=set(df.loc[df.split==split,'id'])
    assert {p.stem for p in (RUN/'images'/split).glob('*.png')}==esperado
    assert {p.stem for p in (RUN/'labels'/split).glob('*.txt')}==esperado

dataset_yaml=RUN/'dataset_yolo_selection_v3.yaml'
dataset_yaml.write_text(yaml.safe_dump({
    'path':str(RUN.resolve()),'train':'images/train','val':'images/val','test':'images/test',
    'names':{0:'satellite_streak'}
},sort_keys=False),encoding='utf-8')
print(json.dumps({
    'status':'DATASET_GPU_LOCAL_APROVADO','run_id':RUN_ID,
    'sha256_manifest':HASH_MANIFEST,'amostras':len(df),
    'pacote_sha256':hash_pacote
},indent=2,ensure_ascii=False))

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.7/45.7 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 72.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.2/66.2 kB 6.6 MB/s eta 0:00:00
Mounted at /content/drive


Restaurando TAR do dataset:   0%|          | 0.00/861M [00:00<?, ?B/s]

Verificando TAR local:   0%|          | 0.00/861M [00:00<?, ?B/s]

Extraindo dataset local:   0%|          | 0/12004 [00:00<?, ?arq/s]

{
  "status": "DATASET_GPU_LOCAL_APROVADO",
  "run_id": "synthetic_v3_c41d5091de56",
  "sha256_manifest": "b635aff10e8b24d3729b4b399f50a8223e0d4373d155a7f16bf232c58d844b42",
  "amostras": 6000,
  "pacote_sha256": "163f46b536aa416c39c806490d6e6066b393bdb53a1e6de5911f24699a4b5434"
}


## 2. Contratos finais e gate pré-teste
Esta seção não executa o modelo no teste. Ela confirma o contrato imutável dos três
checkpoints, o resultado Hough já congelado, a composição exata do split e a ausência
de um evento YOLO final anterior.

In [3]:
from datetime import datetime, timezone

EXPECTED_FINAL_CONTRACT_SHA256='0a8d3537969dcb033dc8cdfa1a9d63dc24de4cc98e212c42fab62621126890a7'
EXPECTED_FINAL_CONFIG_SHA256='457b70c8e96176827bf4a3f755963e6c6c3e607bec7d8133233518b26be27870'
EXPECTED_HOUGH_CONFIG_SHA256='14ef0ea56a2e86bf1581225c4b9a0733d482834e5f7d3fab18b3eaf76f03a2e5'
EXPECTED_HOUGH_RESULT_SHA256='5b476de5a6015b6c38758a43e8c46719050bc036d28362d5f7d43f13e8ffa3b8'

DRIVE_FINAL=BACKUP_DIR/'experiments/yolo_final_multiseed_v3'/RUN_ID
DRIVE_TEST=BACKUP_DIR/'experiments/yolo_protected_test_v3'/RUN_ID
DRIVE_RUN=BACKUP_DIR/'data/synthetic_runs'/RUN_ID
CONTRATO_PATH=DRIVE_FINAL/'contrato_modelos_finais_v3.json'
CONFIG_FINAL_PATH=DRIVE_FINAL/'protocolo_treino_final_multiseed_v3.json'
PONTEIRO_PATH=BACKUP_DIR/'experiments/yolo_final_multiseed_v3/LATEST_FINAL_MODELS_V3.json'
LOCK_PATH=DRIVE_TEST/'TRAVA_EVENTO_TESTE_YOLO_V3.json'

def verificar_hash_canonico(registro,campo_hash):
    copia=dict(registro);observado=copia.pop(campo_hash)
    texto=json.dumps(copia,sort_keys=True,separators=(',',':'))
    calculado=hashlib.sha256(texto.encode()).hexdigest()
    assert calculado==observado,f'Hash canônico divergente em {campo_hash}'
    return observado

contrato=json.loads(CONTRATO_PATH.read_text(encoding='utf-8'))
CONFIG_FINAL_TEST=json.loads(CONFIG_FINAL_PATH.read_text(encoding='utf-8'))
ponteiro=json.loads(PONTEIRO_PATH.read_text(encoding='utf-8'))
HASH_CONTRATO_FINAL=verificar_hash_canonico(contrato,'sha256_contrato_modelos_finais')
HASH_CONFIG_FINAL=verificar_hash_canonico(CONFIG_FINAL_TEST,'sha256_config_final')
assert HASH_CONTRATO_FINAL==EXPECTED_FINAL_CONTRACT_SHA256
assert HASH_CONFIG_FINAL==EXPECTED_FINAL_CONFIG_SHA256==contrato['sha256_config_final']
assert contrato['aprovado'] is True and contrato['run_id']==RUN_ID
assert contrato['sha256_manifest']==HASH_MANIFEST
assert contrato['threshold_confianca']==0.45
assert contrato['seeds_avaliadas']==[20260823,20260824,20260825]
assert contrato['teste_sintetico_lido'] is False and contrato['asta_lido'] is False
assert contrato['plano_teste_protegido']['evento_unico'] is True
assert contrato['plano_teste_protegido']['avaliar_tres_checkpoints'] is True
assert contrato['plano_teste_protegido']['selecionar_seed_com_teste'] is False
assert contrato['plano_teste_protegido']['recalibrar_threshold_com_teste'] is False
assert CONFIG_FINAL_TEST['threshold_confianca_fixo']==contrato['threshold_confianca']==0.45
assert CONFIG_FINAL_TEST['matching']=={'iou':0.3,'angulo':15.0,'distancia':14.0,'cobertura':0.5,'modo':'combinado'}
assert ponteiro['run_id']==RUN_ID
assert ponteiro['sha256_manifest']==HASH_MANIFEST
assert ponteiro['sha256_contrato_modelos_finais']==HASH_CONTRATO_FINAL
assert ponteiro['checkpoint_operacional_seed']==contrato['checkpoint_operacional_seed']==20260824

MODELOS_LOCAL=Path('/content/modelos_finais_yolo_v3')/RUN_ID
MODELOS_LOCAL.mkdir(parents=True,exist_ok=True)
checkpoints=[]
for item in contrato['checkpoints']:
    seed=int(item['seed']);remoto=BACKUP_DIR/item['best_pt_relativo_drive']
    assert remoto.is_file() and sha256_arquivo(remoto)==item['sha256_best_pt']
    local=MODELOS_LOCAL/f'best_seed_{seed}.pt'
    if not local.exists():shutil.copy2(remoto,local)
    assert sha256_arquivo(local)==item['sha256_best_pt']
    checkpoints.append({**item,'seed':seed,'local':str(local)})
assert [x['seed'] for x in checkpoints]==[20260823,20260824,20260825]

hough_dir=DRIVE_RUN/'hough_v3'
hough_config=json.loads((hough_dir/'config_congelada_v3.json').read_text(encoding='utf-8'))
hough_lock=json.loads((hough_dir/'TRAVA_TESTE_FINAL.json').read_text(encoding='utf-8'))
hough_result_path=hough_dir/'resultado_test_v3.csv'
assert hough_config['sha256_config']==EXPECTED_HOUGH_CONFIG_SHA256
hough_sem_hash=dict(hough_config);hough_hash_observado=hough_sem_hash.pop('sha256_config')
hough_hash_calculado=hashlib.sha256(json.dumps(hough_sem_hash,sort_keys=True).encode()).hexdigest()
assert hough_hash_calculado==hough_hash_observado
assert hough_lock['sha256_config']==EXPECTED_HOUGH_CONFIG_SHA256
assert hough_lock['sha256_resultado']==EXPECTED_HOUGH_RESULT_SHA256
assert sha256_arquivo(hough_result_path)==EXPECTED_HOUGH_RESULT_SHA256
assert hough_config['sha256_manifest']==HASH_MANIFEST==hough_lock['sha256_manifest']

df_test=df[df.split=='test'].copy().sort_values('id').reset_index(drop=True)
ids_test=set(df_test.id.astype(str));pasta_test=RUN/'images/test'
assert len(df_test)==900 and (df_test.tipo=='positivo').sum()==450 and (df_test.tipo=='negativo').sum()==450
assert df_test.sha256_fundo.nunique()==40
assert {p.stem for p in pasta_test.glob('*.png')}==ids_test
assert {p.stem for p in (RUN/'labels/test').glob('*.txt')}==ids_test
fundos={s:set(df.loc[df.split==s,'sha256_fundo']) for s in ['train','val','test']}
assert fundos['train'].isdisjoint(fundos['val']) and fundos['train'].isdisjoint(fundos['test'])
assert fundos['val'].isdisjoint(fundos['test'])

lock_existente=json.loads(LOCK_PATH.read_text(encoding='utf-8')) if LOCK_PATH.exists() else None
if lock_existente is None:
    status_pre='TESTE_YOLO_PRONTO_PARA_EVENTO_UNICO'
elif lock_existente.get('status')=='concluido':
    status_pre='BLOQUEADO_TESTE_YOLO_JA_CONCLUIDO'
else:
    status_pre='EVENTO_YOLO_INCOMPLETO_EXIGE_RETOMADA_CONTROLADA'
gate_pre={
    'status':status_pre,'run_id':RUN_ID,'sha256_manifest':HASH_MANIFEST,
    'sha256_contrato_modelos_finais':HASH_CONTRATO_FINAL,
    'sha256_config_final':HASH_CONFIG_FINAL,
    'seeds':[x['seed'] for x in checkpoints],'imagens_test':len(df_test),
    'positivas_test':int((df_test.tipo=='positivo').sum()),
    'negativas_test':int((df_test.tipo=='negativo').sum()),
    'fundos_test':int(df_test.sha256_fundo.nunique()),
    'sha256_hough_config':hough_config['sha256_config'],
    'sha256_hough_resultado':hough_lock['sha256_resultado'],
    'evento_existente':lock_existente
}
print(json.dumps(gate_pre,indent=2,ensure_ascii=False))
if status_pre=='TESTE_YOLO_PRONTO_PARA_EVENTO_UNICO':
    print('Gate pré-teste aprovado. Não habilite o evento antes de compartilhar este relatório.')

{
  "status": "TESTE_YOLO_PRONTO_PARA_EVENTO_UNICO",
  "run_id": "synthetic_v3_c41d5091de56",
  "sha256_manifest": "b635aff10e8b24d3729b4b399f50a8223e0d4373d155a7f16bf232c58d844b42",
  "sha256_contrato_modelos_finais": "0a8d3537969dcb033dc8cdfa1a9d63dc24de4cc98e212c42fab62621126890a7",
  "sha256_config_final": "457b70c8e96176827bf4a3f755963e6c6c3e607bec7d8133233518b26be27870",
  "seeds": [
    20260823,
    20260824,
    20260825
  ],
  "imagens_test": 900,
  "positivas_test": 450,
  "negativas_test": 450,
  "fundos_test": 40,
  "sha256_hough_config": "14ef0ea56a2e86bf1581225c4b9a0733d482834e5f7d3fab18b3eaf76f03a2e5",
  "sha256_hough_resultado": "5b476de5a6015b6c38758a43e8c46719050bc036d28362d5f7d43f13e8ffa3b8",
  "evento_existente": null
}
Gate pré-teste aprovado. Não habilite o evento antes de compartilhar este relatório.


## 3. Funções congeladas de matching e estatística
As funções abaixo reproduzem o matching combinado fixado no Notebook 09. Há um único
GT por imagem positiva; previsões adicionais contam como falsos positivos. O bootstrap
reamostra os 40 fundos e calcula a média de F1 das três seeds em cada réplica.

In [4]:
def parametros_linha(p0,p1):
    p0,p1=np.asarray(p0,dtype=float),np.asarray(p1,dtype=float)
    v=p1-p0;comp=float(np.linalg.norm(v));direcao=v/comp if comp>0 else np.array([1.,0.])
    return (p0+p1)/2,direcao,comp

def centerline(p):
    p=np.asarray(p,dtype=float);d01=np.linalg.norm(p[0]-p[1]);d12=np.linalg.norm(p[1]-p[2])
    return ((p[0]+p[3])/2,(p[1]+p[2])/2) if d01>=d12 else ((p[0]+p[1])/2,(p[2]+p[3])/2)

def iou_poligonos(a,b):
    a,b=np.asarray(a,np.float32),np.asarray(b,np.float32)
    aa=abs(float(cv2.contourArea(a)));ab=abs(float(cv2.contourArea(b)))
    inter=float(cv2.intersectConvexConvex(a,b)[0]);uniao=aa+ab-inter
    return inter/uniao if uniao>0 else 0.0

def metricas_match(pred,gt):
    p0,p1=centerline(pred);g0,g1=centerline(gt)
    cp,dp,_=parametros_linha(p0,p1);cg,dg,lg=parametros_linha(g0,g1)
    ang=float(np.degrees(np.arccos(np.clip(abs(np.dot(dp,dg)),-1,1))))
    v=cp-cg;dist=float(np.linalg.norm(v-np.dot(v,dg)*dg))
    t=sorted([np.dot(p0-cg,dg),np.dot(p1-cg,dg)])
    inter=max(0,min(t[1],lg/2)-max(t[0],-lg/2));cob=float(inter/lg) if lg>0 else 0.0
    return {'iou':iou_poligonos(pred,gt),'angulo':ang,'distancia':dist,'cobertura':cob}

def ler_gt(linha):
    tokens=(RUN/'labels'/linha.split/(linha.id+'.txt')).read_text(encoding='utf-8').split()
    return None if not tokens else np.asarray([float(x) for x in tokens[1:9]]).reshape(4,2)*np.array([512.,512.])

def score_match(m,lim):
    miou=m['iou']>=lim['iou']
    mgeo=(m['angulo']<=lim['angulo'] and m['distancia']<=lim['distancia'] and m['cobertura']>=lim['cobertura'])
    if not (miou or mgeo):return None
    siou=m['iou']/lim['iou'] if miou else -np.inf
    sgeo=((1-m['angulo']/lim['angulo'])+(1-m['distancia']/lim['distancia'])+m['cobertura']) if mgeo else -np.inf
    return max(siou,sgeo)

def avaliar_predicoes(dados,predicoes,threshold):
    saida=[];lim_match={k:CONFIG_FINAL_TEST['matching'][k] for k in ['iou','angulo','distancia','cobertura']}
    for linha in dados.itertuples():
        grupo=predicoes[(predicoes.id==linha.id)&(predicoes.conf>=threshold)].copy()
        candidatos=[]
        for _,g in grupo.iterrows():
            poly=g[['x1','y1','x2','y2','x3','y3','x4','y4']].to_numpy(float).reshape(4,2)
            candidatos.append((poly,float(g.conf)))
        gt=ler_gt(linha);aceitos=[]
        if gt is not None:
            for poly,conf in candidatos:
                met=metricas_match(poly,gt);score=score_match(met,lim_match)
                if score is not None:aceitos.append((score,conf,met))
        melhor=max(aceitos,key=lambda x:(x[0],x[1])) if aceitos else None
        match=melhor is not None
        saida.append({
            'id':linha.id,'split':linha.split,'tipo':linha.tipo,'subtipo':linha.subtipo,
            'fundo_origem':linha.fundo_origem,'sha256_fundo':linha.sha256_fundo,
            'tp':int(gt is not None and match),'fp':max(0,len(candidatos)-int(gt is not None and match)),
            'fn':int(gt is not None and not match),'n_predicoes':len(candidatos),
            'conf_match':melhor[1] if melhor else np.nan,
            'iou_match':melhor[2]['iou'] if melhor else np.nan,
            'angulo_match':melhor[2]['angulo'] if melhor else np.nan,
            'distancia_match':melhor[2]['distancia'] if melhor else np.nan,
            'cobertura_match':melhor[2]['cobertura'] if melhor else np.nan
        })
    return pd.DataFrame(saida)

def metricas_contagens(tp,fp,fn):
    p=tp/(tp+fp) if tp+fp else 0.0;r=tp/(tp+fn) if tp+fn else 0.0
    return p,r,2*p*r/(p+r) if p+r else 0.0

def resumir_resultado(resultado):
    tp=int(resultado.tp.sum());fp=int(resultado.fp.sum());fn=int(resultado.fn.sum())
    p,r,f1=metricas_contagens(tp,fp,fn);neg=resultado[resultado.tipo=='negativo']
    return {'TP':tp,'FP':fp,'FN':fn,'precisao':p,'recall':r,'f1':f1,
            'fp_por_imagem_negativa':float(neg.fp.mean()),
            'taxa_frames_negativos_com_fp':float((neg.fp>0).mean())}

assert abs(metricas_match(np.array([[0,0],[100,0],[100,4],[0,4]],float),np.array([[0,0],[100,0],[100,4],[0,4]],float))['angulo'])<1e-9
print('Funções geométricas e estatísticas congeladas.')

Funções geométricas e estatísticas congeladas.


## 4. Evento único, retomável e protegido
Primeiro execute esta célula com as duas chaves False. Depois de revisar e compartilhar
o gate pré-teste, altere somente EXECUTAR_TESTE_PROTEGIDO=True. Use a chave de retomada
apenas se o mesmo evento tiver sido interrompido e revisado; seeds já concluídas por hash
serão reutilizadas. Um evento concluído é recusado incondicionalmente.

In [5]:
EXECUTAR_TESTE_PROTEGIDO=True
RETOMAR_EVENTO_INCOMPLETO=False

if not EXECUTAR_TESTE_PROTEGIDO:
    print('Evento único bloqueado. Compartilhe primeiro o gate pré-teste da seção 2.')
else:
    import gc,os,uuid,ultralytics
    from ultralytics import YOLO
    assert ultralytics.__version__=='8.4.127'
    DRIVE_TEST.mkdir(parents=True,exist_ok=True)
    def atualizar_lock(evento_atual):
        temporario=LOCK_PATH.with_name(f"{LOCK_PATH.stem}.{evento_atual['event_id']}.tmp")
        temporario.write_text(json.dumps(evento_atual,indent=2),encoding='utf-8')
        os.replace(temporario,LOCK_PATH)
    agora=datetime.now(timezone.utc).isoformat()
    if LOCK_PATH.exists():
        evento=json.loads(LOCK_PATH.read_text(encoding='utf-8'))
        assert evento['sha256_contrato_modelos_finais']==HASH_CONTRATO_FINAL
        assert evento['sha256_config_final']==HASH_CONFIG_FINAL
        assert evento['sha256_manifest']==HASH_MANIFEST
        if evento.get('status')=='concluido':
            raise RuntimeError('Teste YOLO final já concluído. Nova inferência é proibida.')
        if not RETOMAR_EVENTO_INCOMPLETO:
            raise RuntimeError('Há evento incompleto. Revise a trava e habilite somente RETOMAR_EVENTO_INCOMPLETO.')
    else:
        evento={'status':'em_execucao','event_id':str(uuid.uuid4()),'iniciado_em_utc':agora,
                'run_id':RUN_ID,'sha256_manifest':HASH_MANIFEST,
                'sha256_contrato_modelos_finais':HASH_CONTRATO_FINAL,
                'sha256_config_final':HASH_CONFIG_FINAL,
                'seeds_previstas':[x['seed'] for x in checkpoints],'seeds_concluidas':[]}
        LOCK_PATH.parent.mkdir(parents=True,exist_ok=True)
        with LOCK_PATH.open('x',encoding='utf-8') as arq:json.dump(evento,arq,indent=2)

    EVENT_DIR=DRIVE_TEST/f"evento_{evento['event_id']}";EVENT_DIR.mkdir(parents=True,exist_ok=True)
    registros_teste=[]
    for item in checkpoints:
        seed=int(item['seed']);SEED_DRIVE=EVENT_DIR/f'seed_{seed}'
        registro_path=SEED_DRIVE/'conclusao_teste_seed.json'
        if registro_path.exists():
            registro=json.loads(registro_path.read_text(encoding='utf-8'))
            assert registro['sha256_contrato_modelos_finais']==HASH_CONTRATO_FINAL
            assert registro['sha256_checkpoint']==item['sha256_best_pt']
            for nome,campo in [('predicoes_test.csv','sha256_predicoes'),('resultado_test.csv','sha256_resultado')]:
                assert sha256_arquivo(SEED_DRIVE/nome)==registro[campo]
            registros_teste.append(registro);print(f'Seed {seed} já concluída neste evento e reutilizada.')
            continue

        LOCAL_SEED=Path('/content/yolo_protected_test_v3')/evento['event_id']/f'seed_{seed}'
        LOCAL_SEED.mkdir(parents=True,exist_ok=True)
        gc.collect();torch.cuda.empty_cache();modelo=YOLO(item['local'])
        preds=[];tempos=[];ids=[]
        resultados=modelo.predict(source=str(pasta_test),stream=True,batch=1,
            imgsz=CONFIG_FINAL_TEST['treino']['imgsz'],conf=CONFIG_FINAL_TEST['predicao_val']['conf_min'],
            iou=CONFIG_FINAL_TEST['predicao_val']['iou_nms'],max_det=CONFIG_FINAL_TEST['predicao_val']['max_det'],
            device=0,verbose=False)
        for resultado in tqdm(resultados,total=len(ids_test),desc=f'TESTE seed {seed}',unit='img'):
            id_amostra=Path(resultado.path).stem;ids.append(id_amostra)
            tempos.append(float(resultado.speed.get('inference',np.nan)))
            if resultado.obb is None:continue
            for poly,conf in zip(resultado.obb.xyxyxyxy.cpu().numpy(),resultado.obb.conf.cpu().numpy()):
                plano=poly.reshape(-1);preds.append({'id':id_amostra,'conf':float(conf),
                    **{f'{e}{i+1}':float(plano[2*i+(0 if e=='x' else 1)]) for i in range(4) for e in ['x','y']}})
        assert len(ids)==len(ids_test) and len(set(ids))==len(ids) and set(ids)==ids_test
        pred=pd.DataFrame(preds,columns=['id','conf','x1','y1','x2','y2','x3','y3','x4','y4'])
        pred_path=LOCAL_SEED/'predicoes_test.csv';pred.to_csv(pred_path,index=False)
        por_imagem=avaliar_predicoes(df_test,pred,CONFIG_FINAL_TEST['threshold_confianca_fixo'])
        resultado_path=LOCAL_SEED/'resultado_test.csv';por_imagem.to_csv(resultado_path,index=False)
        resumo=resumir_resultado(por_imagem)
        metricas=modelo.val(data=str(dataset_yaml),split='test',imgsz=CONFIG_FINAL_TEST['treino']['imgsz'],
            batch=CONFIG_FINAL_TEST['treino']['batch'],device=0,workers=CONFIG_FINAL_TEST['treino']['workers'],
            plots=False,verbose=False,project=str(LOCAL_SEED/'ultralytics'),name='test',exist_ok=True)
        registro={'protocolo':'yolo_test_protegido_v3','event_id':evento['event_id'],'seed':seed,
            'run_id':RUN_ID,'sha256_manifest':HASH_MANIFEST,
            'sha256_contrato_modelos_finais':HASH_CONTRATO_FINAL,
            'sha256_config_final':HASH_CONFIG_FINAL,
            'sha256_checkpoint':item['sha256_best_pt'],'threshold_confianca':CONFIG_FINAL_TEST['threshold_confianca_fixo'],
            **resumo,'map50_obb':float(metricas.box.map50),'map50_95_obb':float(metricas.box.map),
            'inferencia_ms_media':float(np.nanmean(tempos)),
            'sha256_predicoes':sha256_arquivo(pred_path),'sha256_resultado':sha256_arquivo(resultado_path),
            'concluido_em_utc':datetime.now(timezone.utc).isoformat()}
        registro_local=LOCAL_SEED/'conclusao_teste_seed.json'
        registro_local.write_text(json.dumps(registro,indent=2),encoding='utf-8')
        SEED_DRIVE.mkdir(parents=True,exist_ok=True)
        arquivos_seed=sorted(p for p in LOCAL_SEED.rglob('*') if p.is_file() and p!=registro_local)
        for arquivo in tqdm(arquivos_seed,desc=f'Backup teste seed {seed}',unit='arq'):
            alvo=SEED_DRIVE/arquivo.relative_to(LOCAL_SEED);alvo.parent.mkdir(parents=True,exist_ok=True)
            shutil.copy2(arquivo,alvo)
        # O registro é o commit da seed e só pode existir depois de todos os artefatos.
        shutil.copy2(registro_local,registro_path)
        assert sha256_arquivo(SEED_DRIVE/'predicoes_test.csv')==registro['sha256_predicoes']
        assert sha256_arquivo(SEED_DRIVE/'resultado_test.csv')==registro['sha256_resultado']
        registros_teste.append(registro)
        evento['seeds_concluidas']=sorted({*evento.get('seeds_concluidas',[]),seed})
        atualizar_lock(evento)
        del resultados,resultado,metricas,modelo;gc.collect();torch.cuda.empty_cache()

    assert sorted(r['seed'] for r in registros_teste)==[20260823,20260824,20260825]
    CONS_LOCAL=Path('/content/yolo_protected_test_v3')/evento['event_id']/'consolidado'
    CONS_LOCAL.mkdir(parents=True,exist_ok=True)
    tabelas=[]
    for registro in registros_teste:
        tab=pd.read_csv(EVENT_DIR/f"seed_{registro['seed']}"/'resultado_test.csv');tab['seed']=registro['seed'];tabelas.append(tab)
    resultados_yolo=pd.concat(tabelas,ignore_index=True)
    assert len(resultados_yolo)==2700 and set(resultados_yolo.id)==ids_test
    comparacao_seeds=pd.DataFrame(registros_teste).sort_values('seed')
    colunas_metricas=['precisao','recall','f1','fp_por_imagem_negativa','taxa_frames_negativos_com_fp','map50_obb','map50_95_obb','inferencia_ms_media']
    agregado=comparacao_seeds[colunas_metricas].agg(['mean','std','min','max']).T.reset_index(names='metrica')

    positivos=resultados_yolo[resultados_yolo.tipo=='positivo']
    recall_seed_subtipo=(positivos.groupby(['seed','subtipo'])[['tp','fn']].sum().reset_index())
    recall_seed_subtipo['recall']=recall_seed_subtipo.tp/(recall_seed_subtipo.tp+recall_seed_subtipo.fn)
    recall_subtipo=recall_seed_subtipo.groupby('subtipo').recall.agg(['mean','std','min','max']).reset_index()
    negativos=resultados_yolo[resultados_yolo.tipo=='negativo'].copy()
    neg_seed_subtipo=negativos.groupby(['seed','subtipo']).agg(
        fp_por_imagem=('fp','mean'),taxa_frames_com_fp=('fp',lambda s:float((s>0).mean()))).reset_index()
    neg_subtipo=neg_seed_subtipo.groupby('subtipo')[['fp_por_imagem','taxa_frames_com_fp']].agg(['mean','std','min','max'])
    geometria=(positivos[positivos.tp==1].groupby('seed')[['angulo_match','distancia_match','cobertura_match','iou_match']].mean())

    hough=pd.read_csv(hough_result_path).sort_values('id').reset_index(drop=True)
    assert set(hough.id.astype(str))==ids_test and len(hough)==900
    htp=int(hough.tp_combinado.sum());hfp=int(hough.fp_combinado.sum());hfn=int(hough.fn_combinado.sum())
    hp,hr,hf1=metricas_contagens(htp,hfp,hfn);hneg=hough[hough.tipo=='negativo']
    resumo_hough={'metodo':'Hough congelado','precisao':hp,'recall':hr,'f1':hf1,
        'fp_por_imagem_negativa':float(hneg.fp_combinado.mean()),
        'taxa_frames_negativos_com_fp':float((hneg.fp_combinado>0).mean()),'map50_95_obb':None}
    medias=comparacao_seeds[colunas_metricas].mean()
    resumo_yolo={'metodo':'YOLO11n-OBB média 3 seeds','precisao':float(medias.precisao),
        'recall':float(medias.recall),'f1':float(medias.f1),
        'fp_por_imagem_negativa':float(medias.fp_por_imagem_negativa),
        'taxa_frames_negativos_com_fp':float(medias.taxa_frames_negativos_com_fp),
        'map50_95_obb':float(medias.map50_95_obb)}
    comparacao_metodos=pd.DataFrame([resumo_hough,resumo_yolo])

    fundos_ids=sorted(df_test.sha256_fundo.unique());assert len(fundos_ids)==40
    yolo_fundos={int(seed):g.groupby('sha256_fundo')[['tp','fp','fn']].sum()
                 for seed,g in resultados_yolo.groupby('seed')}
    hough_fundos=hough.groupby('sha256_fundo')[['tp_combinado','fp_combinado','fn_combinado']].sum()
    assert all(set(t.index)==set(fundos_ids) for t in yolo_fundos.values()) and set(hough_fundos.index)==set(fundos_ids)
    rng=np.random.default_rng(20260823);boots=[]
    for _ in tqdm(range(2000),desc='Bootstrap YOLO × Hough por fundo',unit='rep'):
        amostra=rng.choice(fundos_ids,size=len(fundos_ids),replace=True);f1_seeds=[]
        for seed in sorted(yolo_fundos):
            s=yolo_fundos[seed].loc[amostra].sum();f1_seeds.append(metricas_contagens(s.tp,s.fp,s.fn)[2])
        hs=hough_fundos.loc[amostra].sum();hf=metricas_contagens(hs.tp_combinado,hs.fp_combinado,hs.fn_combinado)[2]
        ym=float(np.mean(f1_seeds));boots.append({'f1_yolo_media_seeds':ym,'f1_hough':hf,'delta_f1_yolo_menos_hough':ym-hf})
    bootstrap=pd.DataFrame(boots)
    intervalos={c:[float(x) for x in np.percentile(bootstrap[c],[2.5,97.5])] for c in bootstrap.columns}

    comparacao_seeds.to_csv(CONS_LOCAL/'comparacao_seeds_test_v3.csv',index=False)
    agregado.to_csv(CONS_LOCAL/'variabilidade_seeds_test_v3.csv',index=False)
    recall_subtipo.to_csv(CONS_LOCAL/'recall_por_subtipo_test_v3.csv',index=False)
    neg_subtipo.to_csv(CONS_LOCAL/'negativos_por_subtipo_test_v3.csv')
    geometria.to_csv(CONS_LOCAL/'geometria_matches_test_v3.csv')
    comparacao_metodos.to_csv(CONS_LOCAL/'comparacao_yolo_hough_test_v3.csv',index=False)
    bootstrap.to_csv(CONS_LOCAL/'bootstrap_yolo_hough_por_fundo_v3.csv',index=False)
    resumo_final={'protocolo':'yolo_test_protegido_v3','status':'TESTE_YOLO_PROTEGIDO_CONCLUIDO',
        'event_id':evento['event_id'],'run_id':RUN_ID,'sha256_manifest':HASH_MANIFEST,
        'sha256_contrato_modelos_finais':HASH_CONTRATO_FINAL,
        'sha256_config_final':HASH_CONFIG_FINAL,
        'seeds':[20260823,20260824,20260825],
        'threshold_confianca':CONFIG_FINAL_TEST['threshold_confianca_fixo'],
        'resultado_primario_yolo_media_seeds':resumo_yolo,
        'variabilidade_f1':{'media':float(comparacao_seeds.f1.mean()),'desvio_padrao_amostral':float(comparacao_seeds.f1.std()),
            'min':float(comparacao_seeds.f1.min()),'max':float(comparacao_seeds.f1.max())},
        'hough_congelado':resumo_hough,'bootstrap_por_fundo':{'seed':20260823,'replicacoes':2000,
            'n_fundos':40,'ic95_percentil':intervalos},
        'sha256_tabela_seeds':sha256_arquivo(CONS_LOCAL/'comparacao_seeds_test_v3.csv'),
        'sha256_comparacao_yolo_hough':sha256_arquivo(CONS_LOCAL/'comparacao_yolo_hough_test_v3.csv'),
        'sha256_bootstrap':sha256_arquivo(CONS_LOCAL/'bootstrap_yolo_hough_por_fundo_v3.csv'),
        'selecionou_seed_com_teste':False,'recalibrou_threshold_com_teste':False,'asta_lido':False}
    texto=json.dumps(resumo_final,sort_keys=True,separators=(',',':'))
    resumo_final['sha256_resultado_final']=hashlib.sha256(texto.encode()).hexdigest()
    resumo_path=CONS_LOCAL/'resumo_final_yolo_test_v3.json';resumo_path.write_text(json.dumps(resumo_final,indent=2),encoding='utf-8')
    copiar_arvore_com_progresso(CONS_LOCAL,EVENT_DIR/'consolidado','Backup consolidação teste YOLO')
    assert sha256_arquivo(EVENT_DIR/'consolidado/resumo_final_yolo_test_v3.json')==sha256_arquivo(resumo_path)
    evento.update({'status':'concluido','concluido_em_utc':datetime.now(timezone.utc).isoformat(),
        'seeds_concluidas':[20260823,20260824,20260825],
        'sha256_resultado_final':resumo_final['sha256_resultado_final'],
        'resumo_relativo':str((EVENT_DIR/'consolidado/resumo_final_yolo_test_v3.json').relative_to(BACKUP_DIR))})
    atualizar_lock(evento)
    print(comparacao_seeds[['seed','precisao','recall','f1','taxa_frames_negativos_com_fp','map50_95_obb','inferencia_ms_media']])
    print(comparacao_metodos)
    print(json.dumps(resumo_final,indent=2,ensure_ascii=False))
    print('TESTE YOLO PROTEGIDO CONCLUÍDO E TRAVADO. ASTA permanece intocado.')

Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart#ultralytics-settings.


TESTE seed 20260823:   0%|          | 0/900 [00:00<?, ?img/s]

Ultralytics 8.4.127 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2233.5±1326.3 MB/s, size: 117.3 KB)
val: Scanning /content/data/synthetic_runs/synthetic_v3_c41d5091de56/labels/test... 900 images, 450 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 900/900 1.6Kit/s 0.6s
val: New cache created: /content/data/synthetic_runs/synthetic_v3_c41d5091de56/labels/test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 57/57 6.3it/s 9.0s
                   all        900        450       0.96       0.95      0.982      0.556
Speed: 1.1ms preprocess, 3.1ms inference, 0.0ms loss, 2.4ms postprocess per image


Backup teste seed 20260823:   0%|          | 0/2 [00:00<?, ?arq/s]

TESTE seed 20260824:   0%|          | 0/900 [00:00<?, ?img/s]

...

Ultralytics 8.4.127 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 3387.7±1049.6 MB/s, size: 138.9 KB)
val: Scanning /content/data/synthetic_runs/synthetic_v3_c41d5091de56/labels/test.cache... 900 images, 450 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 900/900 314.6Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 57/57 6.6it/s 8.7s
                   all        900        450      0.946      0.916      0.967      0.627
Speed: 1.0ms preprocess, 2.8ms inference, 0.0ms loss, 2.6ms postprocess per image


Backup teste seed 20260825:   0%|          | 0/2 [00:00<?, ?arq/s]

Bootstrap YOLO × Hough por fundo:   0%|          | 0/2000 [00:00<?, ?rep/s]

Backup consolidação teste YOLO:   0%|          | 0/8 [00:00<?, ?arq/s]

       seed  precisao    recall        f1  taxa_frames_negativos_com_fp  \
0  20260823  0.943844  0.971111  0.957284                           0.0   
1  20260824  0.971239  0.975556  0.973392                           0.0   
2  20260825  0.946581  0.984444  0.965142                           0.0   

   map50_95_obb  inferencia_ms_media  
0      0.556077             9.934360  
1      0.488485             9.944469  
2      0.627022            10.271085  
                      metodo  precisao    recall        f1  \
0            Hough congelado  0.441791  0.328889  0.377070   
1  YOLO11n-OBB média 3 seeds  0.953888  0.977037  0.965273   

   fp_por_imagem_negativa  taxa_frames_negativos_com_fp  map50_95_obb  
0                0.204444                      0.044444           NaN  
1                0.000000                      0.000000      0.557195  
{
  "protocolo": "yolo_test_protegido_v3",
  "status": "TESTE_YOLO_PROTEGIDO_CONCLUIDO",
  "event_id": "29fcf556-5eb4-43fc-8d30-8e3d88edf6c9

## 5. Verificação pós-evento (somente leitura)
Esta célula não executa inferência. Depois do evento, confirma a trava final, o hash do
resumo e imprime o relatório persistido no Drive.

In [6]:
if not LOCK_PATH.exists():
    print('Evento ainda não executado.')
else:
    trava_final=json.loads(LOCK_PATH.read_text(encoding='utf-8'))
    print(json.dumps(trava_final,indent=2,ensure_ascii=False))
    if trava_final.get('status')=='concluido':
        resumo_remoto=BACKUP_DIR/trava_final['resumo_relativo']
        resumo=json.loads(resumo_remoto.read_text(encoding='utf-8'))
        copia=dict(resumo);observado=copia.pop('sha256_resultado_final')
        calculado=hashlib.sha256(json.dumps(copia,sort_keys=True,separators=(',',':')).encode()).hexdigest()
        assert calculado==observado==trava_final['sha256_resultado_final']
        assert resumo['sha256_contrato_modelos_finais']==EXPECTED_FINAL_CONTRACT_SHA256
        assert resumo['sha256_config_final']==EXPECTED_FINAL_CONFIG_SHA256
        assert resumo['selecionou_seed_com_teste'] is False and resumo['recalibrou_threshold_com_teste'] is False
        assert resumo['asta_lido'] is False
        print(json.dumps(resumo,indent=2,ensure_ascii=False))
        print('BACKUP TESTE YOLO PROTEGIDO APROVADO.')

{
  "status": "concluido",
  "event_id": "29fcf556-5eb4-43fc-8d30-8e3d88edf6c9",
  "iniciado_em_utc": "2026-08-30T01:46:04.235743+00:00",
  "run_id": "synthetic_v3_c41d5091de56",
  "sha256_manifest": "b635aff10e8b24d3729b4b399f50a8223e0d4373d155a7f16bf232c58d844b42",
  "sha256_contrato_modelos_finais": "0a8d3537969dcb033dc8cdfa1a9d63dc24de4cc98e212c42fab62621126890a7",
  "sha256_config_final": "457b70c8e96176827bf4a3f755963e6c6c3e607bec7d8133233518b26be27870",
  "seeds_previstas": [
    20260823,
    20260824,
    20260825
  ],
  "seeds_concluidas": [
    20260823,
    20260824,
    20260825
  ],
  "concluido_em_utc": "2026-08-30T01:47:40.579108+00:00",
  "sha256_resultado_final": "878f63dbedea0f72a4b7ed77c238d49cb2b17a89cfce4f51a27ea3b4d7fa0485",
  "resumo_relativo": "experiments/yolo_protected_test_v3/synthetic_v3_c41d5091de56/evento_29fcf556-5eb4-43fc-8d30-8e3d88edf6c9/consolidado/resumo_final_yolo_test_v3.json"
}
{
  "protocolo": "yolo_test_protegido_v3",
  "status": "TESTE_YOLO_PR